In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, cohen_kappa_score, roc_auc_score
)

print("TF version:", tf.__version__)

RAW_DATASET_PATH = "../data/raw_dataset"
LABELS_STEP3 = "../ExpertAnnotations/step3_labels.csv"

# label mapping (severity)
MAP_3 = {
    "No Hypoxia (Normal)": 0,
    "Mild Hypoxia (Suspicious)": 1,
    "Severe Hypoxia (Pathological)": 2,
}
UNINT = "Uninterpretable (Filtered)"


c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TF version: 2.16.1


In [3]:
# Import your function from wherever you placed it
# Example assumes: preprocessing/preprocessing.py contains process_dataset_from_folder
from dataPreprocessing import process_dataset_from_folder

outputs = process_dataset_from_folder(RAW_DATASET_PATH)

print("Records:", len(outputs["rec_ids"]))
print("Valid step1:", sum(outputs["valid_mask"]["step1"]))
print("Example step4 shape:", outputs["step4"][0].shape)


Records: 223
Valid step1: 0
Example step4 shape: (2, 1800, 1)


In [25]:
def build_step_dataset(outputs, labels_csv, step_name="step4"):
    labels = pd.read_csv(labels_csv)
    labels["rec_id"] = labels["rec_id"].astype(str)

    X_list = outputs[step_name]
    valid = outputs["valid_mask"][step_name]
    rec_ids = [str(r) for r in outputs["rec_ids"]]

    X = []
    y_sev = []      # severity: 0/1/2, or -1 for uninterpretable
    y_interp = []   # interpretable: 1 if interpretable else 0
    kept = []

    for rid, x, ok in zip(rec_ids, X_list, valid):
        if (not ok) or (x is None):
            continue

        row = labels[labels["rec_id"] == rid]
        if len(row) == 0:
            continue

        lbl = row.iloc[0]["Clinical_Label"]

        if lbl == UNINT:
            y_sev.append(-1)
            y_interp.append(0)
        else:
            y_sev.append(MAP_3[lbl])
            y_interp.append(1)

        X.append(x)
        kept.append(rid)

    X = np.stack(X).astype(np.float32)          # (N,2,1800,1)
    y_sev = np.array(y_sev, dtype=np.int32)     # (N,)
    y_interp = np.array(y_interp, dtype=np.int32)  # (N,)

    return X, y_sev, y_interp, kept


X4, y4_sev, y4_interp, ids4 = build_step_dataset(outputs, LABELS_STEP3, "step3")

print("X4:", X4.shape)
print("Severity label counts (including -1):", {k:int(v) for k,v in zip(*np.unique(y4_sev, return_counts=True))})
print("Interpretability counts:", {k:int(v) for k,v in zip(*np.unique(y4_interp, return_counts=True))})


X4: (223, 2, 1800, 1)
Severity label counts (including -1): {-1: 70, 0: 63, 1: 56, 2: 34}
Interpretability counts: {0: 70, 1: 153}


In [26]:
def make_ordinal_targets(y_sev_int, num_classes=3):
    """
    CORAL targets: shape (N, K-1)
    For K=3, output dim=2
    """
    K = num_classes
    y = y_sev_int.copy()
    y = np.clip(y, 0, K-1)  # safe for uninterpretable placeholders
    # (y > 0), (y > 1), ...
    ord_targets = np.stack([(y > t).astype(np.float32) for t in range(K-1)], axis=1)
    return ord_targets

y4_ord = make_ordinal_targets(y4_sev, num_classes=3)           # (N,2)
y4_interp_f = y4_interp.astype(np.float32).reshape(-1, 1)      # (N,1)

# sample weights: severity gets weight 0 when y_sev == -1
w_sev = (y4_sev != -1).astype(np.float32)     # (N,)
w_interp = np.ones_like(w_sev, dtype=np.float32)

print("Ordinal target shape:", y4_ord.shape)
print("Severity weights nonzero:", int(w_sev.sum()), "out of", len(w_sev))


Ordinal target shape: (223, 2)
Severity weights nonzero: 153 out of 223


In [27]:
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, GlobalAveragePooling2D, Dense, Concatenate
)
from tensorflow.keras.models import Model

def build_ctg_ms_cnn_ordinal_multitask(input_shape=(2, 1800, 1), dropout_rate=0.25):
    inputs = Input(shape=input_shape)

    # --- Stage 1 ---
    x = Conv2D(filters=4, kernel_size=(1, 3), padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)

    # --- Stage 2 ---
    x = DepthwiseConv2D(kernel_size=(2, 1), depth_multiplier=2, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D(pool_size=(1, 4))(x)
    x = Dropout(dropout_rate)(x)

    # --- Multi-scale ---
    b_small = SeparableConv2D(filters=8, kernel_size=(1, 3), padding="same", use_bias=False)(x)
    b_small = BatchNormalization()(b_small); b_small = Activation("relu")(b_small)

    b_med = SeparableConv2D(filters=8, kernel_size=(1, 7), padding="same", use_bias=False)(x)
    b_med = BatchNormalization()(b_med); b_med = Activation("relu")(b_med)

    b_large = SeparableConv2D(filters=8, kernel_size=(1, 15), padding="same", use_bias=False)(x)
    b_large = BatchNormalization()(b_large); b_large = Activation("relu")(b_large)

    x = Concatenate(axis=-1)([b_small, b_med, b_large])

    x = Conv2D(filters=8, kernel_size=(1, 1), use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D(pool_size=(1, 4))(x)
    x = Dropout(dropout_rate)(x)

    x = GlobalAveragePooling2D()(x)

    # --- Heads ---
    # Ordinal severity: K-1 sigmoid outputs (for K=3 → 2 outputs)
    severity_ord = Dense(2, activation="sigmoid", name="severity_ord")(x)

    # Interpretability: 1 sigmoid
    interpretable = Dense(1, activation="sigmoid", name="interpretable")(x)

    return Model(inputs, [severity_ord, interpretable], name="CTG-MS-CNN-Ordinal-MT")


In [33]:
def ordinal_bce_loss(y_true, y_pred):
    # y_true: (N, K-1), y_pred: (N, K-1)
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=-1)

model = build_ctg_ms_cnn_ordinal_multitask()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=[ordinal_bce_loss, "binary_crossentropy"],
    loss_weights=[1.0, 0.5],
)


model.summary()


Model: "CTG-MS-CNN-Ordinal-MT"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 2, 1800,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 2, 1800,   │         12 │ input_layer_1[0]… │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2, 1800,   │         16 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_1  │ (None, 1, 1800,   │         16 │ batch_normalizat… │
│ (DepthwiseConv2D)   │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 1800,   │         32 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 1, 1800,   │          0 │ batch_normalizat… │
│ (Activation)        │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d_2 │ (None, 1, 450, 8) │          0 │ activation_5[0][… │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 1, 450, 8) │          0 │ average_pooling2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_3  │ (None, 1, 450, 8) │         88 │ dropout_2[0][0]   │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_4  │ (None, 1, 450, 8) │        120 │ dropout_2[0][0]   │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_5  │ (None, 1, 450, 8) │        184 │ dropout_2[0][0]   │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 450, 8) │         32 │ separable_conv2d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 450, 8) │         32 │ separable_conv2d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 450, 8) │         32 │ separable_conv2d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 1, 450, 8) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 1, 450, 8) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 1, 450, 8) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                 

 Total params: 815 (3.18 KB)

 Trainable params: 727 (2.84 KB)

 Non-trainable params: 88 (352.00 B)

In [32]:
print("X4:", type(X4), X4.shape, X4.dtype)
print("y4_ord:", type(y4_ord), y4_ord.shape, y4_ord.dtype)
print("y4_interp_f:", type(y4_interp_f), y4_interp_f.shape, y4_interp_f.dtype)
print("w_sev:", type(w_sev), w_sev.shape, w_sev.dtype, "nonzero:", int(w_sev.sum()))
print("w_interp:", type(w_interp), w_interp.shape, w_interp.dtype)

print("model.output_names:", model.output_names)
print("targets keys:", ["severity_ord", "interpretable"])


X4: <class 'numpy.ndarray'> (223, 2, 1800, 1) float32
y4_ord: <class 'numpy.ndarray'> (223, 2) float32
y4_interp_f: <class 'numpy.ndarray'> (223, 1) float32
w_sev: <class 'numpy.ndarray'> (223,) float32 nonzero: 153
w_interp: <class 'numpy.ndarray'> (223,) float32
model.output_names: ListWrapper(['severity_ord', 'interpretable'])
targets keys: ['severity_ord', 'interpretable']


In [34]:
w_sev = np.asarray(w_sev).reshape(-1).astype(np.float32)
w_interp = np.asarray(w_interp).reshape(-1).astype(np.float32)

y4_ord = np.asarray(y4_ord).astype(np.float32)
y4_interp_f = np.asarray(y4_interp_f).astype(np.float32)  # shape (N,1) is fine


In [35]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="loss", patience=25, restore_best_weights=True, verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="loss", factor=0.5, patience=15, min_lr=1e-7, verbose=1
)

print("model.output_names:", model.output_names)  # should be ['severity_ord','interpretable']

history = model.fit(
    X4,
    [y4_ord, y4_interp_f],
    sample_weight=[w_sev, w_interp],
    batch_size=8,
    epochs=300,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)



model.output_names: ListWrapper(['severity_ord', 'interpretable'])
Epoch 1/300


28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - interpretable_loss: 0.9436 - loss: 0.9923 - severity_ord_loss: 0.5207 - learning_rate: 0.0010
Epoch 2/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - interpretable_loss: 0.8471 - loss: 0.9182 - severity_ord_loss: 0.4940 - learning_rate: 0.0010
Epoch 3/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - interpretable_loss: 0.7875 - loss: 0.8673 - severity_ord_loss: 0.4744 - learning_rate: 0.0010
Epoch 4/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - interpretable_loss: 0.7268 - loss: 0.8254 - severity_ord_loss: 0.4613 - learning_rate: 0.0010
Epoch 5/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - interpretable_loss: 0.6973 - loss: 0.7934 - severity_ord_loss: 0.4453 - learning_rate: 0.0010
Epoch 6/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - interpretable_loss: 0.6702 - loss: 0.7667 - severity_ord_loss: 0.4316 - learning_rate: 0.0010
Epoch 7/300
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - interpretable_loss: 0.6522 - loss: 0.7531 - severity_ord_loss: 0.4270 - learn

In [36]:
def decode_ordinal_to_class(y_ord_pred, thr=0.5):
    # y_ord_pred: (N,2). Class = count of outputs >= thr
    return (y_ord_pred >= thr).sum(axis=1).astype(int)

sev_pred_ord, interp_pred = model.predict(X4, verbose=0)

yhat_sev = decode_ordinal_to_class(sev_pred_ord, thr=0.5)
yhat_interp = (interp_pred.reshape(-1) >= 0.5).astype(int)

# Evaluate interpretability on ALL samples
print("=== Interpretability (ALL) ===")
print("Acc:", accuracy_score(y4_interp, yhat_interp))
print("F1:", f1_score(y4_interp, yhat_interp, zero_division=0))

# Evaluate severity only where interpretable (y_sev != -1)
mask = (y4_sev != -1)
print("\n=== Severity (interpretable only) ===")
print("N:", mask.sum())

print("Acc:", accuracy_score(y4_sev[mask], yhat_sev[mask]))
print("Macro-F1:", f1_score(y4_sev[mask], yhat_sev[mask], average="macro", zero_division=0))
print("QWK:", cohen_kappa_score(y4_sev[mask], yhat_sev[mask], weights="quadratic"))
print("\nReport:\n", classification_report(y4_sev[mask], yhat_sev[mask], zero_division=0))
print("Confusion:\n", confusion_matrix(y4_sev[mask], yhat_sev[mask]))


=== Interpretability (ALL) ===
Acc: 0.7130044843049327
F1: 0.8212290502793296

=== Severity (interpretable only) ===
N: 153
Acc: 0.40522875816993464
Macro-F1: 0.24239921066519812
QWK: 0.0024494794856092517

Report:
               precision    recall  f1-score   support

           0       0.41      0.89      0.56        63
           1       0.35      0.11      0.16        56
           2       0.00      0.00      0.00        34

    accuracy                           0.41       153
   macro avg       0.25      0.33      0.24       153
weighted avg       0.30      0.41      0.29       153

Confusion:
 [[56  7  0]
 [50  6  0]
 [30  4  0]]


In [ ]:
mask = (y4_sev != -1)  # or y3_sev for step3
print("Interpretable N:", mask.sum())
print("Severity class counts:", dict(zip(*np.unique(y4_sev[mask], return_counts=True))))
print("w_sev nonzero:", int(w_sev.sum()))


Interpretable N: 153
Severity class counts: {0: 63, 1: 56, 2: 34}
w_sev nonzero: 153


: 